In [1]:
import numpy as np
import dendropy

In [2]:
from skbio import DistanceMatrix
from skbio.tree import nj
from dendropy.calculate import treecompare

In [3]:
import skbio.diversity.beta

In [4]:
def get_distance_matrix(tree):
    taxa = list(tree.taxon_namespace)
    pdc = tree.phylogenetic_distance_matrix()
    n = len(taxa)
    dist_matrix = np.zeros((n, n))
    for i, t1 in enumerate(taxa):
        for j, t2 in enumerate(taxa):
            if i != j:
                dist_matrix[i, j] = pdc(t1, t2)
    return dist_matrix

In [5]:
from simulation.datagen import rand_dataset
# Generate dataset
# 10, 4, 5
# 20, 8, 10
data = rand_dataset(10, 4, 5)
tree = data['tree']


In [6]:
# Print initial tree in Newick format
initial_tree_newick = tree.as_string(schema="newick")
# print("Initial Tree in Newick format:")
# print(initial_tree_newick)

In [7]:
# Calculate the initial distance matrix
initial_dist_matrix = get_distance_matrix(tree)
# initial_dist_matrix

In [8]:
# Create a DistanceMatrix for scikit-bio
# NOTE: this was wrong, the command `leaf_node_iter` was returning taxa in the order they were added to the tree, not in the order they appear in the Newick string (which is what tree.taxa_namespace returns)
# ids_new = [leaf.taxon.label for leaf in tree.leaf_node_iter()]
ids_new = [l.label for l in tree.taxon_namespace]
print(ids_new)
print(tree.taxon_namespace)
dm_skbio = DistanceMatrix(initial_dist_matrix, ids_new)

['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10']
['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10']


In [9]:
# Perform NJ algorithm using scikit-bio
nj_tree = nj(dm_skbio)

In [10]:
# Get NJ tree in Newick format
nj_tree_newick = str(nj_tree)
print("NJ Tree in Newick format:")
print(nj_tree_newick)


NJ Tree in Newick format:
(T5:0.167404,(T3:0.218241,(T7:0.359906,((T2:1.342953,(T9:0.097413,T6:0.097413):1.245541):4.821943,(T1:0.076136,(T8:0.0,T4:0.0):0.076136):0.986326):0.702556):0.141665):0.050837,T10:0.167404);



In [11]:
# Convert NJ tree Newick string to scikit-bio TreeNode
nj_tree_skbio = skbio.TreeNode.read([nj_tree_newick])

In [12]:
# NOTE: do not create a new one, use the one from the already existing tree (which is the one used for the distance matrix)
taxon_namespace = tree.taxon_namespace

In [13]:
# Convert initial tree Newick string to scikit-bio TreeNode
initial_tree_skbio = skbio.TreeNode.read([initial_tree_newick])

In [14]:
# Convert initial tree Newick string to DendroPy Tree using the same TaxonNamespace
initial_tree_dendropy = dendropy.Tree.get(data=initial_tree_newick, schema="newick", taxon_namespace=taxon_namespace)


In [15]:
# Convert NJ tree Newick string to DendroPy Tree using the same TaxonNamespace
nj_tree_dendropy = dendropy.Tree.get(data=nj_tree_newick, schema="newick", taxon_namespace=taxon_namespace)


In [16]:
# Compute the RF distance using DendroPy
rf_distance = treecompare.symmetric_difference(initial_tree_dendropy, nj_tree_dendropy)
print("RF Distance:", rf_distance)


RF Distance: 7


In [17]:
# Generate ASCII art for the initial tree
print("Initial Tree (ASCII art):")
initial_tree_dendropy.print_plot()

Initial Tree (ASCII art):
                                             /----------------------------- T2 
/--------------------------------------------+                                 
|                                            |              /-------------- T9 
|                                            \--------------+                  
|                                                           \-------------- T6 
|                                                                              
+                                                           /-------------- T4 
|                                            /--------------+                  
|              /-----------------------------+              \-------------- T8 
|              |                             |                                 
|              |                             \----------------------------- T1 
|              |                                                               
\-------------

In [18]:
# Generate ASCII art for the NJ tree
print("NJ Tree (ASCII art):")
nj_tree_dendropy.print_plot()

NJ Tree (ASCII art):
/-------------------------------------------------------------------------- T5 
|                                                                              
|           /-------------------------------------------------------------- T3 
|           |                                                                  
|-----------+            /------------------------------------------------- T7 
|           |            |                                                     
|           |            |                        /------------------------ T2 
|           \------------+           /------------+                            
|                        |           |            |           /------------ T9 
+                        |           |            \-----------+                
|                        \-----------+                        \------------ T6 
|                                    |                                         
|                  

In [19]:
rf_distance = treecompare.symmetric_difference(initial_tree_dendropy, initial_tree_dendropy)
print("RF Distance:", rf_distance)


RF Distance: 0
